# LoMa EDisGo-Workshop 13.2.2025

Contents:
1. Topology Setup
2. Worst Case Time Series Creation
3. Grid Investigation
4. Results
5. Additional Time Series


In [ ]:
%load_ext jupyter_black

In [ ]:
import os
import requests
import sys

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from copy import deepcopy
from numpy.random import default_rng
from pathlib import Path

from edisgo import EDisGo
from edisgo.io.db import engine
from edisgo.tools.logger import setup_logger

In [ ]:
# Nur wegen der Übersicht. Normalerweise nicht zu empfehlen
import warnings

warnings.filterwarnings("ignore")

## 1 Topology Setup

In this section we load all components into a newly created edisgo object. This includes the lines, buses, transformers, switches, generators, loads, heat pumps and battery storages.

### Standard components

Set up a new edisgo object:

In [ ]:
conf_path = Path.home() / "Downloads" / "egon-data.configuration.yaml"
db_engine = engine(path=conf_path, ssh=True)
ding0_grid = Path.home() / ".edisgo" / "husum_grids" / "35725"

edisgo = EDisGo(ding0_grid=ding0_grid, legacy_ding0_grids=False, engine=db_engine)

The ding0 grids are not up to date and their capacity is not sufficient for the connected loads and generators. To update the imported grids they need to be extended first with the function ```reinforce()```.

Grids are reinforced for their worst case scenarios. The corresponding time series are created with ```set_time_series_worst_case_analysis()```. 

In [ ]:
edisgo.set_time_series_worst_case_analysis()

In [ ]:
edisgo.reinforce()

### Plot grid topology (MV)

The topology can be visualized with the ```plot_mv_grid_topology()```. For ```technologies=True``` the buses sizes and colors are determined to the type and size of the technologies connected to it. 

- red: nodes with substation secondary side
- light blue: nodes distribution substations's primary side
- green: nodes with fluctuating generators
- black: nodes with conventional generators
- grey: disconnecting points
- dark blue: branch trees

In [ ]:
sizes_dict = {
    "BranchTee": 10000,
    "GeneratorFluctuating": 100000,
    "Generator": 100000,
    "Load": 100000,
    "LVStation": 50000,
    "MVStation": 120000,
    "Storage": 100000,
    "DisconnectingPoint": 75000,
    "else": 200000,
}

sizes_dict = {k: v / 10 for k, v in sizes_dict.items()}

In [ ]:
edisgo.plot_mv_grid_topology(technologies=True, sizes_dict=sizes_dict)

### Topology-Module Data Structure

Let's get familiar with the topology module:

In [ ]:
edisgo.topology.generators_df[["p_nom", "type"]].groupby("type").sum()

In [ ]:
edisgo.topology.loads_df[["p_set", "type"]].groupby("type").sum()

Number of LV grids in the MV grid

In [ ]:
len(list(edisgo.topology.mv_grid.lv_grids))

Total number of lines:

In [ ]:
len(edisgo.topology.lines_df.index)

Number of lines in one of the low voltage grids.

In [ ]:
len(edisgo.topology.grids[5].lines_df.index)

### Basic components addition and removal

To see how a loaded network can be adapted later on, we add a solar plant to a random bus.

Components can also be added according to their geolocation with the function integrate_component_based_on_geolocation().

In [ ]:
edisgo.topology.generators_df

Add a generator with the function ```add_component()``` or ```add_generator()```. 

In [ ]:
rng = default_rng(1)
rnd_bus = rng.choice(edisgo.topology.buses_df.index, size=1)[0]
generator_type = "solar"

new_generator = edisgo.add_component(
    comp_type="generator", p_nom=0.01, bus=rnd_bus, generator_type=generator_type
)

In [ ]:
edisgo.topology.generators_df

We can also add a heat pump:

In [ ]:
edisgo.topology.loads_df

In [ ]:
new_load = edisgo.add_component(
    comp_type="load", p_set=0.01, bus=rnd_bus, type="heat_pump"
)

In [ ]:
edisgo.topology.loads_df

Single components can be removed with ```remove_component()```

In [ ]:
edisgo.remove_component(comp_type="generator", comp_name=new_generator)
edisgo.remove_component(comp_type="load", comp_name=new_load)

In [ ]:
edisgo.topology.generators_df

In [ ]:
edisgo.topology.loads_df

### Add flexible components to grid 

For realistic future grids we also add further components like additional generators, home batteries, (charging points) and heat pumps. The components are added according to the scenario "eGon2035" and the data from the oedb.

In [ ]:
scenario = "eGon2035"

In [ ]:
# copy the edisgo object for later comparisons
edisgo_orig = deepcopy(edisgo)

In [ ]:
# Retry if running into "Connection reset by peer" error
edisgo = deepcopy(edisgo_orig)

edisgo.import_generators(generator_scenario=scenario)
edisgo.import_home_batteries(scenario=scenario)
edisgo.import_heat_pumps(scenario=scenario)

In [ ]:
# This takes too long for the workshop, but needs to be mentioned
# edisgo_obj.import_dsm(scenario=scenario)
# edisgo_obj.import_electromobility(
#     data_source="oedb", scenario=scenario
#)

In [ ]:
edisgo.topology.generators_df[["p_nom", "type"]].groupby("type").sum()

In [ ]:
edisgo_orig.topology.generators_df[["p_nom", "type"]].groupby("type").sum()

In [ ]:
edisgo.topology.loads_df[["p_set", "type"]].groupby("type").sum()

In [ ]:
edisgo_orig.topology.loads_df[["p_set", "type"]].groupby("type").sum()

In [ ]:
edisgo.topology.storage_units_df["p_nom"].sum()

In [ ]:
edisgo_orig.topology.storage_units_df["p_nom"].sum()

## 2 Worst Case Time Series Creation

Create timeseries for the four worst cases MV load case, LV load case, MV feed-in case, LV feed-in case with the function  set_time_series_worst_case_analysis().

In conventional grid expansion planning worst-cases, the heavy load flow and the reverse power flow, are used to determine grid expansion needs. eDisGo allows you to analyze these cases separately or together. Choose between the following options:

* **’feed-in_case’** 
  
  Feed-in and demand for the worst-case scenario "reverse power flow" are generated (e.g. conventional electricity demand is set to 15% of maximum demand for loads connected to the MV grid and 10% for loads connected to the LV grid and feed-in of all generators is set to the nominal power of the generator, except for PV systems where it is by default set to 85% of the nominal power)

  
* **’load_case’**

  Feed-in and demand for the worst-case scenario "heavy load flow" are generated (e.g. demand of all conventional loads is by default set to maximum demand and feed-in of all generators is set to zero)


* **[’feed-in_case’, ’load_case’]**

  Both cases are set up.
  
By default both cases are set up.

Feed-in and demand in the two worst-cases are defined in the [config file 'config_timeseries.cfg'](https://edisgo.readthedocs.io/en/latest/configs.html#config-timeseries) and can be changed by setting different values in the config file. 

In [ ]:
edisgo.set_time_series_worst_case_analysis()

The function creates time series for four time steps since both worst cases are defined seperately for the LV and the MV grid with individual simultanerity factors.

In [ ]:
edisgo.timeseries.timeindex_worst_cases

In [ ]:
edisgo.timeseries.loads_active_power.loc[
    edisgo.timeseries.timeindex_worst_cases["load_case_mv"]
]

## 3 Grid Investigation

Execute a power flow analysis to determine line overloads and voltage deviations for the MV load case timeseries with the function analyze():

In [ ]:
edisgo.analyze(timesteps=edisgo.timeseries.timeindex_worst_cases["load_case_mv"])

A geoplot wtih the bus and line colrs based on the voltage deviations and line loadings repectively can be created with ```plot_mv_line_loading()```.

In [ ]:
edisgo.plot_mv_line_loading(
    node_color="voltage_deviation",
    timestep=edisgo.timeseries.timeindex_worst_cases["load_case_mv"],
)

For a better overview of the voltage deviations and line loads in the entire grid, edisgo provides histrogram plots.

In [ ]:
edisgo.histogram_voltage(binwidth=0.005)

In [ ]:
edisgo.histogram_relative_line_load(binwidth=0.1)

## 4 Results

Now we reinforce the fully equipped grid. For a shorter runtime, only the MV grid is considered by setting ```mode = "mv"```.

In [ ]:
edisgo.reinforce()

In [ ]:
edisgo.plot_mv_line_loading(
    node_color="voltage_deviation",
    timestep=edisgo.timeseries.timeindex_worst_cases["load_case_mv"],
)

In [ ]:
# edisgo.analyze() 

In [ ]:
edisgo.histogram_voltage(binwidth=0.005)

In [ ]:
edisgo.histogram_relative_line_load(binwidth=0.1)

The module ```results```holds the outpts of the reinforcement

In [ ]:
# The equipment changes of the reinforcement after the grid setup have to be dropped
edisgo.results.equipment_changes[len(edisgo_orig.results.equipment_changes) :].head()

In [ ]:
edisgo.results.grid_expansion_costs[
    len(edisgo_orig.results.grid_expansion_costs) :
].head()

### TODO: Visualisierung der Ergebnisse

In [ ]:
# edisgo_costs = deepcopy(edisgo)
# edisgo_costs.results.grid_expansion_costs = edisgo.results.grid_expansion_costs[len(edisgo_orig.results.grid_expansion_costs) :]
# edisgo_costs.plot_mv_grid_expansion_costs()

## 5 Additional Time Series

* show time resolution